# RAG Project: Recipe recommendations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandroviajando/ir/blob/wip/7_rag_project.ipynb)

## Setup

In [1]:
try:
    import datasets
except ImportError:
    ! pip -q install datasets
    import datasets

import json

Load the `recipes` dataset with the [datasets](https://huggingface.co/docs/datasets/access) library:

In [2]:
! if [ ! -f "./data/recipes.parquet" ]; then mkdir -p "./data" && curl -L -o ./data/recipes.parquet https://people.cs.kuleuven.be/~thomas.bauwens/irse_documents_2025_recipes.parquet; fi

dataset = datasets.load_dataset("parquet", data_files="./data/recipes.parquet")["train"]

print("One document:")
for example in dataset:
    for k, v in example.items():
        print(f"'{k}' = {v}\n")
    break

One document:
'name' = arriba baked winter squash mexican style

'ingredients' = winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt

'steps' = make a choice and proceed with recipe, depending on size of squash , cut into half or fourths, remove seeds, for spicy squash , drizzle olive oil or melted butter over each cut squash piece, season with mexican seasoning mix ii, for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece, season with sweet mexican spice mix, bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin, be careful not to burn the squash especially if you opt to use sugar or butter, if you feel more comfortable , cover the squash with aluminum foil the first half hour , give or take , of baking, if desired , season with salt

'tags' = 60-minutes-or-less, time-to-make, course, main-ingredient, cuisine, preparation, occasion, north-american, side-

Load the `queries.json` file, which contains the gold queries - which can be used to debug the retriever and estimate mean average precision (MAP):

In [3]:
! if [ ! -f "./data/queries.json" ]; then mkdir -p "./data" && curl -L -o ./data/queries.json https://people.cs.kuleuven.be/~thomas.bauwens/irse_queries_2025_recipes.json; fi

queries = json.load(open("./data/queries.json", "r"))

print(queries["queries"][0])

{'q': 'What temperature should I pre-heat my oven to when making chicken quesadillas?', 'r': [[167945, 1], [167954, 1], [21548, 1], [218187, 1], [168524, 1], [68174, 1], [34390, 1], [34410, 1], [85623, 1], [46749, 1], [83613, 1], [210101, 1], [192707, 1], [19157, 1], [46809, 1], [168697, 1], [139022, 1], [168732, 1], [151851, 1], [45356, 1], [45357, 1], [40241, 1], [6453, 1], [179511, 1], [168270, 1], [19788, 1], [223067, 1], [19339, 1], [98716, 1], [191431, 1], [25072, 1]], 'a': '375 is a good temperature, but can go as low as 350 or as high as 400. Adjust times accordingly (longer for lower temperatures).'}


You can see that the `queries` dictionary object contains a list of dictionaries, consisting of query (`q`), answer (`a`), and relevant documents (`r`) fields. The integer values in `r` correspond to the `official_id` field in the `recipes.parquet` dataset (see above), along with a relevance score.



Once your TF-IDF model has been implemented and fit on the recipes dataset, you can experiment with retrieving the k-most relevant documents for the queries provided below:

In [4]:
sample_queries = [
    "a cajun style gumbo with an easy roux",
    "I am feeling like eating shrimp tacos tonight. What's a good recipe?",
    "recipe for easy vegetarian lasagna",
    "How do I make spageti and meatballs?",
    "15 minute lunch recipe",
    "Give me suggestion for some easy vegetarian weeknight dinner recipes"
]

For a given query and set of relevant documents, you are also required to create a prompt that instructs a model to complete a certain task (e.g. recipe recommendation). You should experiment with formatting the prompt, as language models have been shown to be sensitive to the exact verbiage of instructions.

In [5]:
prompt = f"""

YOUR PROMPT GOES HERE

"""

In [6]:
irrelevant_context = """
Richard Gary Brautigan (January 30, 1935 – c. September 16, 1984)
was an American novelist, poet, and short story writer. A prolific writer,
he wrote throughout his life and published ten novels, two collections of
short stories, and four books of poetry. Brautigan's work has been published
both in the United States and internationally throughout Europe, Japan,
and China. He is best known for his novels Trout Fishing in America (1967),
In Watermelon Sugar (1968), and The Abortion: An Historical Romance 1966 (1971).
"""

Before loading a model from the HuggingFace hub, you will likely want to create an account at https://huggingface.co/ so that you can get an [**access token**](https://huggingface.co/docs/hub/security-tokens) for models which require identification before usage.

- On your personal machine, you can input this access token by running `huggingface-cli login` in a terminal window.

- In Colab, click the icon in the left sidebar that looks like a key, and *Add a secret* called `HF_TOKEN`.

If you don't do this, you risk running into a "Cannot access gated repo" error.

In [7]:
try:
    from google.colab import userdata

    userdata.get("HF_TOKEN")
except Exception:
    print("Not in Google Colab environment. Set HuggingFace access token with `huggingface-cli login`.")

Not in Google Colab environment. Set HuggingFace access token with `huggingface-cli login`.


In [8]:
try:
    import transformers
except ImportError:
    ! pip -q install git+https://github.com/huggingface/transformers
    ! pip -q install datasets bitsandbytes accelerate xformers einops

import torch
import transformers
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using {device = }.")

Using device = 'mps'.


## 0. LLM without RAG

Quantizing the model via `bitesandbytes` ensures that it doesn't take up too much memory and makes inference more efficient.

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

if device == "mps":
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto"
    )
elif device == "cuda":
    bnb_config = transformers.BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        device_map="auto"
    )

A tokenizer is required in order to convert strings into integer sequences that can be passed as input to the model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

input_string = prompt + sample_queries[0]

encoded_prompt = tokenizer(input_string, return_tensors="pt", add_special_tokens=False)
encoded_prompt = encoded_prompt.to(device)

This is the final generation step, where a forward pass must be made through the entire model. Since the model is large (even after quantization), it might take a while.

In [ ]:
generated_ids = model.generate(**encoded_prompt, max_new_tokens=1000, do_sample=True)
decoded = tokenizer.batch_decode(generated_ids)
print(decoded[0])

We can see that, even without additional context and reference documents, the model is able to generate very coherent recipe instructions. Now, it is up to you to experiment with the RAG framework and see if you can further improve the quality of the model's generation with relevant documents. Refer to the assignment handout for the exact questions we expect you to answer.  

## 1. Architecture

```txt
+------------------+
|  User Query (q)  |
+------------------+
         |
         v
+----------------------------------------------------+
| Query Preprocessing                                |
| - Lowercasing                                      |
| - Tokenization                                     |
| - Stopword removal                                 |
| - Phrase detection (e.g., bigrams, named entities) |
+----------------------------------------------------+
         |
         v
+-------------------------------------+
| Query Encoder fQ(q) (TF-IDF vector) |
+-------------------------------------+
         |
         v
+-------------------------------------------------------------------------------+
| Document Encoder fD(D) (TF-IDF over all docs) → matrix of shape (N docs × H)  |
+-------------------------------------------------------------------------------+
         |
         v
+-------------------------------------------+
| Retrieval Module (NearestNeighbors search)|
| - Compute cosine similarity               |
| - Retrieve top-k most relevant docs       |
+-------------------------------------------+
         |
         v
+---------------------------------------------------------+
| Prompt Formatter I(q, [d̂1, ..., d̂k])                    |
| - Inserts retrieved docs + query into a prompt template |
+---------------------------------------------------------+
         |
         v
+-------------------------------------+
| Generative LM (e.g., Mistral)       |
| - Instruction-following LM          |
| - Outputs natural language answer ŷ |
+-------------------------------------+
```

| Component | Role |
| --------- | ---- |
| User Query (q) | The natural language input from the user asking a question about the dataset. |
| Query Preprocessing | Text normalization (e.g., lowercasing, punctuation removal), plus optional phrase extraction. |
| Query Encoder (fQ) | Converts the query into a vector using the same method as document encoding (e.g., TF-IDF). |
| Document Encoder (fD) | Converts all documents into fixed-size vectors to enable efficient similarity search. |
| Retrieval Module | Computes similarity scores (e.g., cosine) between the query and documents, returns top-k. |
| Prompt Formatter (I) | Assembles a textual prompt that includes query + retrieved docs to condition the LM. |
| Generative LM | Produces the final natural-language output based on the structured prompt. |

- Why preprocess query and documents similarly? → To ensure compatible vector space for retrieval.
- Why format the prompt carefully? → Instruction-tuned LMs rely heavily on how you present the input.
- Why TF-IDF here? → Simple, interpretable, and performant enough for a baseline in RAG.


## 2. Term Vocabulary

TODO:

- stopwords
- multi-word expressions
- size of vocabulary: are rare words worth storing?
- don't combine into one column but pre-process the relevant columns (for next part)

In [ ]:
def preprocess_tags(tags):
  return tags.replace("-", " ").replace(",", "")

dataset = dataset.map(lambda example: {"term_vocab": example["name"] + " " + preprocess_tags(example["tags"])})

print(dataset[0])

RAKE identifies important phrases by analyzing word frequency and co-occurrence while ignoring stopwords. It’s unsupervised and doesn’t require training data.

In [ ]:
! pip install rake-nltk
from rake_nltk import Rake

rake = Rake()

def extract_keywords_rake(text):
    rake.extract_keywords_from_text(text)
    return ' '.join(rake.get_ranked_phrases())

dataset_processed = dataset.map(
    lambda example: {'term_vocab': extract_keywords_rake(example['name'] + ' ' + preprocess_tags(example['tags']))}
)

Gensim’s Phrases uses pointwise mutual information (PMI) to detect common multi-word expressions across the corpus, which is more robust than n-grams.

In [ ]:
! pip install gensim
from gensim.models import Phrases
from gensim.models.phrases import Phraser

# Tokenize first
tokenized = [ (example['name'] + ' ' + preprocess_tags(example['tags'])).lower().split() for example in dataset ]

# Train Phrases model
phrases = Phrases(tokenized, min_count=2, threshold=5)
phraser = Phraser(phrases)

def extract_phrases_gensim(text):
    tokens = text.lower().split()
    return ' '.join(phraser[tokens])

dataset_processed = dataset.map(
    lambda example: {'term_vocab': extract_phrases_gensim(example['name'] + ' ' + preprocess_tags(example['tags']))}
)

spaCy extracts noun phrases and named entities using syntactic structure and context — great for recipes or any structured text.

In [ ]:
! pip install spacy
! python -m spacy download en_core_web_sm

import spacy
nlp = spacy.load("en_core_web_sm")

def extract_phrases_spacy(text):
    doc = nlp(text)
    phrases = set(chunk.text for chunk in doc.noun_chunks)
    phrases.update(ent.text for ent in doc.ents)
    return ' '.join(phrases)

dataset_processed = dataset.map(
    lambda example: {'term_vocab': extract_phrases_spacy(example['name'] + ' ' + preprocess_tags(example['tags']))}
)

| Approach | Customizable | Speed | Good for |
| -------- | ------------ | ----- | -------- |
| RAKE     | Medium       | Fast  | Keywords and phrases |
| Gensim Phrases | High   | Medium | Repeated word combos |
| spaCy    | High         | Slower | Grammar-based phrases |

## 3. Document Embeddings

- run experiments which fields to include in embeddings
- After embedding the documents di as TF-IDF vectors di, what mechanism does your implementation use to derive query vectors? (Be precise!)
- How does your mechanism handle queries that don’t contain any words of your vocabulary?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(dataset["term_vocab"])

## 4. Retrieval

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn_model = NearestNeighbors(n_neighbors=5, metric='cosine')
nn_model.fit(tfidf_matrix)

In [ ]:
query_vector = vectorizer.transform(["a cajun style gumbo with an easy roux"])
# Cosine similarity: lower distance value indicates higher similarity
distances, indices = nn_model.kneighbors(query_vector)
relevant_document_ids = dataset["official_id"][indices.tolist()[0]]

In [ ]:
distances.tolist()[0], indices.tolist()[0], int(indices[0][0])

In [ ]:
dataset[indices.tolist()[0][0]]